# Car Sales Price Prediction using PyTorch

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

print("Libraries loaded")


Libraries loaded


# Load and preprocess data

In [ ]:

# Load dataset
df = pd.read_csv("car_purchasing.csv", encoding='ISO-8859-1')
df.head()


,customer name,customer e-mail,country,gender,age,annual Salary,credit card debt,net worth,car purchase amount
0,Martina Avila,cubilia.Curae.Phasellus@quisaccumsanconvallis.edu,Bulgaria,0,41.851720,62812.09301,11609.380910,238961.2505,35321.45877
1,Harlan Barnes,eu.dolor@diam.co.uk,Belize,0,40.870623,66646.89292,9572.957136,530973.9078,45115.52566
2,Naomi Rodriquez,vulputate.mauris.sagittis@ametconsectetueradip...,Algeria,1,43.152897,53798.55112,11160.355060,638467.1773,42925.70921
3,Jade Cunningham,malesuada@dignissim.com,Cook Islands,1,58.271369,79370.03798,14426.164850,548599.0524,67422.36313
4,Cedric Leach,felis.ullamcorper.viverra@egetmollislectus.net,Brazil,1,57.313749,59729.15130,5358.712177,560304.0671,55915.46248


In [ ]:
# Select features and target using correct column names from the dataset
X = df[['gender', 'age', 'annual Salary', 'credit card debt', 'net worth']].copy()
y = df['car purchase amount'].copy()

# Normalize features
scaler_x = MinMaxScaler()
X_scaled = scaler_x.fit_transform(X)

# Normalize target
scaler_y = MinMaxScaler()
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1))

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test)

# DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# Define ANN model in PyTorch

In [ ]:
class CarPriceModel(nn.Module):
    def __init__(self):
        super(CarPriceModel, self).__init__()
        self.fc1 = nn.Linear(5, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
          input_data = x
          layer1_output = self.fc1(input_data)
          layer1_output = self.relu(layer1_output)

          layer2_output = self.fc2(layer1_output)
          layer2_output = self.relu(layer2_output)

          output = self.fc3(layer2_output)

          return output


model = CarPriceModel()
print(model)


CarPriceModel(
  (fc1): Linear(in_features=5, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
)


# Train the model

In [ ]:
loss_fn= nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

epochs = 100
for epoch in range(epochs):
    for xb, yb in train_loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")


Epoch 10/100, Loss: 0.0001
Epoch 20/100, Loss: 0.0000
Epoch 30/100, Loss: 0.0000
Epoch 40/100, Loss: 0.0000
Epoch 50/100, Loss: 0.0000
Epoch 60/100, Loss: 0.0000
Epoch 70/100, Loss: 0.0000
Epoch 80/100, Loss: 0.0000
Epoch 90/100, Loss: 0.0000
Epoch 100/100, Loss: 0.0000


# Evaluate the model

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

model.eval()

with torch.inference_mode():

    y_pred = model(X_test_tensor)

    mae = mean_absolute_error(
        y_test_tensor.numpy(),
        y_pred.numpy()
    )

    mse = mean_squared_error(
        y_test_tensor.numpy(),
        y_pred.numpy()
    )



    r2 = r2_score(
        y_test_tensor.numpy(),
        y_pred.numpy()
    )

print("MAE:", mae)
print("MSE:", mse)
print("R2 Score:", r2)

MAE: 0.000689324748236686
MSE: 6.661317684120149e-07
RMSE: 0.000816168958250689
R2 Score: 0.9999688863754272
